In [8]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel, wilcoxon
import matplotlib.pyplot as plt


ALPHA = 0.05

# Load file
df = pd.read_csv("vnd_results.csv")

df

,instance,order,best_known,cost,delta_percent,time_seconds
0,N-tiw56r67_250,order1,5289658,5181358,2.047391,2.901623
1,N-tiw56r67_250,order2,5289658,5194932,1.790777,1.692997
2,N-tiw56r66_150,order1,1940755,1905311,1.826300,0.173024
3,N-tiw56r66_150,order2,1940755,1899565,2.122370,0.137035
4,N-tiw56n58_250,order1,2906657,2835798,2.437818,1.753564
...,...,...,...,...,...,...
151,N-t70f11xx_250,order2,13586135,13356888,1.687360,1.523841
152,N-t59b11xx_250,order1,8411910,8221740,2.260723,2.146499
153,N-t59b11xx_250,order2,8411910,8284120,1.519156,1.664862
154,N-be75tot_250,order1,30958609,30312371,2.087426,2.006362


In [2]:
# Fastest order 

quality = df.pivot(index="instance", columns="order", values="delta_percent").reset_index()
quality.columns.name = None
quality = quality.dropna(subset=["order1", "order2"]).copy()

xq = quality["order1"].to_numpy()
yq = quality["order2"].to_numpy()
diff_q = xq - yq   # negative => order1 better

mean_q1 = np.mean(xq)
mean_q2 = np.mean(yq)

tq_stat, tq_p = ttest_rel(xq, yq)
if np.allclose(diff_q, 0):
    wq_stat, wq_p = np.nan, 1.0
else:
    wq_stat, wq_p = wilcoxon(xq, yq, zero_method="wilcox", alternative="two-sided")

best_quality_order = "order1" if mean_q1 < mean_q2 else ("order2" if mean_q2 < mean_q1 else "tie")


print("=== VND: Solution quality ===")
print(f"Mean delta order1: {mean_q1:.6f}")
print(f"Mean delta order2: {mean_q2:.6f}")
print(f"Best quality order: {best_quality_order}")

=== VND: Solution quality ===
Mean delta order1: 2.063339
Mean delta order2: 1.995886
Best quality order: order2


In [3]:
# ----- Time pairing -----
time_df = df.pivot(index="instance", columns="order", values="time_seconds").reset_index()
time_df.columns.name = None
time_df = time_df.dropna(subset=["order1", "order2"]).copy()

xt = time_df["order1"].to_numpy()
yt = time_df["order2"].to_numpy()
diff_t = xt - yt   # negative => order1 faster

mean_t1 = np.mean(xt)
mean_t2 = np.mean(yt)
sum_t1 = np.sum(xt)
sum_t2 = np.sum(yt)

tt_stat, tt_p = ttest_rel(xt, yt)
if np.allclose(diff_t, 0):
    wt_stat, wt_p = np.nan, 1.0
else:
    wt_stat, wt_p = wilcoxon(xt, yt, zero_method="wilcox", alternative="two-sided")

fastest_order = "order1" if mean_t1 < mean_t2 else ("order2" if mean_t2 < mean_t1 else "tie")

print("=== VND: Computation time ===")
print(f"Mean time order1: {mean_t1:.6f} s")
print(f"Mean time order2: {mean_t2:.6f} s")
print(f"Sum time order1 : {sum_t1:.6f} s")
print(f"Sum time order2 : {sum_t2:.6f} s")
print(f"Fastest order: {fastest_order}")

=== VND: Computation time ===
Mean time order1: 1.211202 s
Mean time order2: 0.899855 s
Sum time order1 : 94.473762 s
Sum time order2 : 70.188714 s
Fastest order: order2


In [4]:
# ------------------------------------------------------------
# Per-instance comparison table
# ------------------------------------------------------------
quality = df.pivot(index="instance", columns="order", values="delta_percent").reset_index()
quality.columns.name = None

time_tbl = df.pivot(index="instance", columns="order", values="time_seconds").reset_index()
time_tbl.columns.name = None

comparison_table = quality.merge(time_tbl, on="instance", suffixes=("_delta", "_time"))
comparison_table = comparison_table.rename(columns={
    "order1_delta": "order1_delta_percent",
    "order2_delta": "order2_delta_percent",
    "order1_time": "order1_time_seconds",
    "order2_time": "order2_time_seconds",
})

comparison_table["better_quality"] = comparison_table.apply(
    lambda row: "order1" if row["order1_delta_percent"] < row["order2_delta_percent"]
    else ("order2" if row["order2_delta_percent"] < row["order1_delta_percent"] else "tie"),
    axis=1
)

comparison_table["faster_order"] = comparison_table.apply(
    lambda row: "order1" if row["order1_time_seconds"] < row["order2_time_seconds"]
    else ("order2" if row["order2_time_seconds"] < row["order1_time_seconds"] else "tie"),
    axis=1
)

comparison_table["diff_delta_order1_minus_order2"] = (
    comparison_table["order1_delta_percent"] - comparison_table["order2_delta_percent"]
)
comparison_table["diff_time_order1_minus_order2"] = (
    comparison_table["order1_time_seconds"] - comparison_table["order2_time_seconds"]
)
comparison_table

,instance,order1_delta_percent,order2_delta_percent,order1_time_seconds,order2_time_seconds,better_quality,faster_order,diff_delta_order1_minus_order2,diff_time_order1_minus_order2
0,N-be75eec_150,1.786077,2.105559,0.209203,0.146360,order1,order2,-0.319482,0.062843
1,N-be75eec_250,1.795001,1.784713,2.654522,1.824807,order2,order2,0.010288,0.829715
2,N-be75np_150,1.874615,1.913281,0.197832,0.134004,order1,order2,-0.038666,0.063828
3,N-be75np_250,1.852485,2.094878,2.004866,1.534640,order1,order2,-0.242393,0.470226
4,N-be75oi_150,1.260431,1.708899,0.237568,0.130128,order1,order2,-0.448468,0.107440
...,...,...,...,...,...,...,...,...,...
73,N-tiw56r66_250,2.154226,1.948442,1.930474,1.686233,order2,order2,0.205784,0.244241
74,N-tiw56r67_150,1.910670,2.268489,0.179281,0.137677,order1,order2,-0.357819,0.041604
75,N-tiw56r67_250,2.047391,1.790777,2.901623,1.692997,order2,order2,0.256614,1.208626
76,N-tiw56r72_150,2.075142,2.952590,0.208251,0.113932,order1,order2,-0.877448,0.094319


In [7]:
paired = df.pivot(index="instance", columns="order", values="delta_percent").reset_index()

paired 

order,instance,order1,order2
0,N-be75eec_150,1.786077,2.105559
1,N-be75eec_250,1.795001,1.784713
2,N-be75np_150,1.874615,1.913281
3,N-be75np_250,1.852485,2.094878
4,N-be75oi_150,1.260431,1.708899
...,...,...,...
73,N-tiw56r66_250,2.154226,1.948442
74,N-tiw56r67_150,1.910670,2.268489
75,N-tiw56r67_250,2.047391,1.790777
76,N-tiw56r72_150,2.075142,2.952590


In [ ]:
# Extract paired samples
x = paired["order1"].to_numpy()
y = paired["order2"].to_numpy()

# Descriptive statistics
mean_order1 = np.mean(x)
mean_order2 = np.mean(y)
diff = x - y   # negative => order1 better, positive => order2 better

print("\n=== Descriptive statistics ===")
print(f"Number of common instances: {len(paired)}")
print(f"Mean delta order1: {mean_order1:.6f}")
print(f"Mean delta order2: {mean_order2:.6f}")
print(f"Mean difference (order1 - order2): {np.mean(diff):.6f}")
print(f"Median difference (order1 - order2): {np.median(diff):.6f}")

if mean_order1 < mean_order2:
    print("Better average solution quality: order1")
elif mean_order2 < mean_order1:
    print("Better average solution quality: order2")
else:
    print("Same average solution quality")

# Paired t-test
t_stat, t_pvalue = ttest_rel(x, y)

# Wilcoxon signed-rank test
if np.allclose(diff, 0):
    w_stat = np.nan
    w_pvalue = 1.0
else:
    w_stat, w_pvalue = wilcoxon(x, y, zero_method="wilcox", alternative="two-sided")

print("\n=== Statistical hypothesis tests ===")
print(f"Alpha = {ALPHA}")

print("\nPaired t-test")
print(f"t-statistic = {t_stat:.6f}")
print(f"p-value     = {t_pvalue:.6f}")
if t_pvalue < ALPHA:
    print("Reject H0: significant difference between order1 and order2")
else:
    print("Do not reject H0: no statistically significant difference")

print("\nWilcoxon signed-rank test")
print(f"statistic = {w_stat}")
print(f"p-value   = {w_pvalue:.6f}")
if w_pvalue < ALPHA:
    print("Reject H0: significant difference between order1 and order2")
else:
    print("Do not reject H0: no statistically significant difference")

#If you want a compact sentence for the report, use this too:

if mean_order1 < mean_order2:
    better = "order1"
else:
    better = "order2"

print("\n=== Report sentence ===")
print(
    f"On the common instances, {better} achieves the lower average relative deviation. "
    f"The paired t-test gives p = {t_pvalue:.6f} and the Wilcoxon signed-rank test gives "
    f"p = {w_pvalue:.6f}. At alpha = {ALPHA}, "
    f"{'the difference is statistically significant' if (t_pvalue < ALPHA or w_pvalue < ALPHA) else 'the difference is not statistically significant'}."
)



=== Descriptive statistics ===
Number of common instances: 78
Mean delta order1: 2.063339
Mean delta order2: 1.995886
Mean difference (order1 - order2): 0.067453
Median difference (order1 - order2): -0.005187
Better average solution quality: order2

=== Statistical hypothesis tests ===
Alpha = 0.05

Paired t-test
t-statistic = 1.254192
p-value     = 0.213566
Do not reject H0: no statistically significant difference

Wilcoxon signed-rank test
statistic = 1377.0
p-value   = 0.415443
Do not reject H0: no statistically significant difference

=== Report sentence ===
On the common instances, order2 achieves the lower average relative deviation. The paired t-test gives p = 0.213566 and the Wilcoxon signed-rank test gives p = 0.415443. At alpha = 0.05, the difference is not statistically significant.
